## Loading the data — part 1: statistics

This notebook uses four files, split across two folders in the repo, so they're uploaded in two steps. First, upload the three Eurostat CSVs from `Geospacific/data/processed/`: household internet access, individual internet use, and digital skills. Hold Ctrl (Cmd on Mac) in the file dialog to select all three at once. This notebook only uses `digital_skills_by_age.csv`, but all three are uploaded for consistency with the household internet access notebook.

In [ ]:
from google.colab import files

uploaded_csv = files.upload()
# Select all three from data/processed/:
# - household_internet_access.csv
# - individual_internet_use.csv
# - digital_skills_by_age.csv

print("Uploaded:", list(uploaded_csv.keys()))

## Loading the data — part 2: country boundaries

Next, upload the country boundary map. This file lives in a different folder, `Geospacific/data/geo/`, so it needs a separate upload step: `europe_countries_boundaries.geojson` (European country boundaries from Eurostat/GISCO, supplemented with Kosovo from OpenStreetMap — see the project README for details). **Note:** this file was corrected this session (Kosovo's boundary ring direction was flipped to fix a rendering bug) — re-upload a fresh copy if you have an older one saved locally.

In [ ]:
uploaded_geo = files.upload()
# Select from data/geo/:
# - europe_countries_boundaries.geojson

print("Uploaded:", list(uploaded_geo.keys()))

## Joining the statistics to the map

Statistics and country boundaries are joined on country code (`geo_code` in the data, `CNTR_ID` on the map). This dataset has an `age_group` column (all individuals, 16-24, 25-54, 55-74), used later to build the age-group map. The check below shows which records have no matching shape on the map — expected to be EU/euro-area aggregate values, which have no boundary of their own.

In [ ]:
import pandas as pd
import json

df_raw = pd.read_csv('digital_skills_by_age.csv')

with open('europe_countries_boundaries.geojson') as f:
    geojson = json.load(f)

geo_ids = {f['properties']['CNTR_ID'] for f in geojson['features']}
csv_ids = set(df_raw['geo_code'])
print("In CSV but not on the map:", csv_ids - geo_ids)  # expected: EU aggregates (EU27_2020, EA)

## Digital skills by age group

One map, with a 3x4 grid of buttons below it (rows = years 2021/2023/2025, columns = age groups: all individuals, 16-24, 25-54, 55-74). Clicking a button jumps straight to that combination's coloring, with no animated transition - year and age group aren't a continuous sequence, so animating between them would misleadingly imply a trend that isn't there.

Color scale is purple (`Purples`), to keep this map visually distinct from the internet access ones. Missing data is a distinct amber for any country/year/age-group combination with no confirmed reading (this includes the UK, which is on the map but never reported to this particular Eurostat survey) - no forward-filling and no blending between known values here, since a 2-year gap in this sparser dataset is too big a jump to assume nothing changed in between.

In [ ]:
import plotly.graph_objects as go

# Shared color for "no data" - used for the amber overlay trace below.
MISSING_COLOR = 'rgb(246,217,168)'

AGE_GROUPS = ['All individuals (16-74)', '16-24', '25-54', '55-74']
AGE_LABELS = {'All individuals (16-74)': 'All (16-74)', '16-24': '16-24', '25-54': '25-54', '55-74': '55-74'}
YEARS = sorted(df_raw['year'].unique())

# Fixed country order: every country on the map, not just the ones with
# rows in this dataset. The UK, for example, is on the map but never
# reported to this survey - it should show as "tracked, no data" (amber),
# not fall off the map entirely as "not tracked".
#
# Kosovo is deliberately EXCLUDED here and given its own pair of traces
# further down. Its shape sits almost entirely inside Serbia's (GISCO draws
# Serbia with no hole cut out for Kosovo), and within a single trace Plotly
# draws countries in list order - last one on top - but resolves hover by
# walking the same list and stopping at the FIRST shape under the cursor.
# So whichever of the two is listed last is visible but un-hoverable, and
# whichever is first is hoverable but hidden underneath. No ordering fixes
# both. A separate, later trace does: later traces draw on top AND take
# hover priority.
countries_age = sorted(c for c in geo_ids if c != 'XK')
country_name = {f['properties']['CNTR_ID']: f['properties']['NAME_ENGL'] for f in geojson['features']}

# Precompute, for every (year, age_group) combination: the color values for
# all tracked countries, AND separately the subset with no data at all (so
# a button click can update both the data trace and the gray overlay trace).
combo_data = {}
for year in YEARS:
    for age in AGE_GROUPS:
        subset = df_raw[(df_raw['year'] == year) & (df_raw['age_group'] == age)]
        values = subset.set_index('geo_code')['pct_basic_or_above_digital_skills']
        z = [values.get(c) for c in countries_age]
        hover = [
            f"{country_name.get(c, c)}<br>{v:.1f}%" if pd.notna(v) else f"{country_name.get(c, c)}<br>No data"
            for c, v in zip(countries_age, z)
        ]
        # missing_z/missing_hover are the SAME LENGTH as countries_age (None/""
        # for countries that DO have data) - kept constant across every button
        # click so the amber overlay trace's 'locations' never has to change.
        # A shrinking/growing locations array is what caused Kosovo's hover to
        # show Serbia's info once Kosovo got real data (see Digitalization.ipynb).
        missing_z = [1 if pd.isna(v) else None for v in z]
        missing_hover = [
            f"{country_name.get(c, c)}<br>No data yet" if pd.isna(v) else ""
            for c, v in zip(countries_age, z)
        ]
        # Kosovo's own single-country values, for its dedicated traces.
        xk_v = values.get('XK')
        xk_has = pd.notna(xk_v)
        combo_data[(year, age)] = {
            'z': z, 'hover': hover, 'missing_z': missing_z, 'missing_hover': missing_hover,
            'xk_z': [xk_v if xk_has else None],
            'xk_hover': [f"{country_name.get('XK', 'XK')}<br>{xk_v:.1f}%" if xk_has else ""],
            'xk_missing_z': [None if xk_has else 1],
            'xk_missing_hover': ["" if xk_has else f"{country_name.get('XK', 'XK')}<br>No data yet"]
        }

n_combos_missing = sum(sum(1 for v in d['missing_z'] if v is not None) for d in combo_data.values())
print(f"Combinations built: {len(combo_data)} (years x age groups). Missing data points: {n_combos_missing}")

In [ ]:
# Ocean and "world we don't track" get their own colors, kept separate
# from MISSING_COLOR (the amber overlay trace below) - otherwise Kaliningrad,
# Russia, Belarus etc. would look exactly like a tracked country with
# missing data.
OCEAN_COLOR = 'rgb(223,242,232)'
UNTRACKED_COLOR = 'rgb(211,209,199)'

default_year, default_age = YEARS[0], AGE_GROUPS[0]
default = combo_data[(default_year, default_age)]

fig_age = go.Figure()

# Trace 0: the actual data, colored on the Purples scale.
fig_age.add_trace(go.Choropleth(
    locations=countries_age,
    z=default['z'],
    geojson=geojson,
    featureidkey='properties.CNTR_ID',
    colorscale='Purples',
    zmin=0, zmax=100,
    text=default['hover'],
    hoverinfo='text',
    marker_line_color='white',
    marker_line_width=0.5,
    colorbar=dict(title='% with basic+<br>digital skills')
))

# Trace 1: solid-amber overlay for tracked countries with no data for this
# combination - kept separate from the untracked-world landcolor below.
# 'locations' is the full, constant countries_age list (not just the
# missing subset) - see the comment in the combo_data cell for why.
fig_age.add_trace(go.Choropleth(
    locations=countries_age,
    z=default['missing_z'],
    geojson=geojson,
    featureidkey='properties.CNTR_ID',
    colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]],
    showscale=False,
    text=default['missing_hover'],
    hoverinfo='text',
    marker_line_color='white',
    marker_line_width=0.3
))

# Traces 2 and 3: Kosovo's data and amber overlay, added last so they draw
# on top of Serbia and win hover priority (see the countries_age comment).
fig_age.add_trace(go.Choropleth(
    locations=['XK'],
    z=default['xk_z'],
    geojson=geojson,
    featureidkey='properties.CNTR_ID',
    colorscale='Purples',
    zmin=0, zmax=100,
    showscale=False,
    text=default['xk_hover'],
    hoverinfo='text',
    marker_line_color='white',
    marker_line_width=0.5
))

fig_age.add_trace(go.Choropleth(
    locations=['XK'],
    z=default['xk_missing_z'],
    geojson=geojson,
    featureidkey='properties.CNTR_ID',
    colorscale=[[0, MISSING_COLOR], [1, MISSING_COLOR]],
    showscale=False,
    text=default['xk_missing_hover'],
    hoverinfo='text',
    marker_line_color='white',
    marker_line_width=0.3
))

# Continental-Europe framing. domain.y leaves room at the BOTTOM of the
# figure for the 3x4 button grid (grid goes under the map, not above it).
fig_age.update_geos(
    visible=False,
    lonaxis_range=[-25, 45],
    lataxis_range=[33, 72],
    showland=True,
    landcolor=UNTRACKED_COLOR,
    showocean=True,
    oceancolor=OCEAN_COLOR,
    projection_type='equirectangular',  # clean rectangular edge instead of a
                                         # curved/ragged one at the lon/lat crop
    domain=dict(y=[0.23, 1])  # map takes up more vertical space, button grid
                               # sits closer underneath instead of a big gap
)

fig_age.update_layout(
    title=f'Basic digital skills in Europe — {default_year}, {AGE_LABELS[default_age]}',
    width=1000,
    height=850,
    dragmode=False
)

# Small color-swatch legend explaining the map's background colors (ocean,
# untracked-world land, tracked-but-missing gray) - these aren't part of the
# main Purples color scale, so they need their own key.
LEGEND_ITEMS = [(OCEAN_COLOR, 'Ocean'), (UNTRACKED_COLOR, 'Not tracked'), (MISSING_COLOR, 'No data yet')]
legend_shapes = []
legend_annotations = []
_start_x, _y_pos, _box_w, _gap = 0.02, 0.995, 0.018, 0.13
for i, (color, label) in enumerate(LEGEND_ITEMS):
    x0 = _start_x + i * _gap
    legend_shapes.append(dict(
        type='rect', xref='paper', yref='paper',
        x0=x0, x1=x0 + _box_w, y0=_y_pos - 0.035, y1=_y_pos,
        fillcolor=color, line=dict(color='rgba(0,0,0,0.3)', width=0.5)
    ))
    legend_annotations.append(dict(
        text=label, xref='paper', yref='paper',
        x=x0 + _box_w + 0.006, y=_y_pos - 0.017,
        xanchor='left', yanchor='middle', showarrow=False, font=dict(size=10)
    ))

# One row of 4 buttons (age groups) per year, stacked vertically below the
# map. Each button restyles BOTH traces at once (data + gray overlay),
# updates the title, AND resets the OTHER two rows' "active" button to none
# - otherwise each row remembers its own last click independently, and up
# to 3 buttons end up highlighted at once even though only one combination
# is actually showing.
updatemenus = []
row_labels = []
row_y_positions = [0.17, 0.10, 0.03]  # top to bottom, one per year - tucked
                                       # right under the map now

for row_i, year in enumerate(YEARS):
    buttons = []
    for age in AGE_GROUPS:
        d = combo_data[(year, age)]
        layout_update = {'title': f'Basic digital skills in Europe — {year}, {AGE_LABELS[age]}'}
        for other_row in range(len(YEARS)):
            if other_row != row_i:
                layout_update[f'updatemenus[{other_row}].active'] = -1
        buttons.append(dict(
            label=AGE_LABELS[age],
            method='update',
            args=[
                {
                    'z': [d['z'], d['missing_z'], d['xk_z'], d['xk_missing_z']],
                    'text': [d['hover'], d['missing_hover'], d['xk_hover'], d['xk_missing_hover']]
                },
                layout_update
            ]
        ))
    updatemenus.append(dict(
        type='buttons',
        direction='right',
        buttons=buttons,
        x=0.18, xanchor='left',
        y=row_y_positions[row_i], yanchor='middle',
        pad=dict(t=2, r=2, b=2, l=2),
        showactive=True,
        active=0 if row_i == 0 else -1  # only 2021/All highlighted on load,
                                         # matching what the map shows first
    ))
    # Row label (year) to the left of each button row
    row_labels.append(dict(
        text=f"<b>{year}</b>", xref='paper', yref='paper',
        x=0.08, y=row_y_positions[row_i], xanchor='right', yanchor='middle',
        showarrow=False, font=dict(size=12)
    ))

fig_age.update_layout(updatemenus=updatemenus, annotations=row_labels + legend_annotations, shapes=legend_shapes)

fig_age.show()

## Country code legend

Reference table for every country code used in the map above.

In [ ]:
legend_df = pd.DataFrame({
    'Code': countries_age,
    'Country': [country_name.get(c, c) for c in countries_age]
})
legend_df